# Stage 1: Apply hotfix and validate in Colab

This notebook replaces **all local Anaconda/PowerShell commands**. It only edits the seven Stage 1 hotfix source files (with backups) and writes validation reports. It does **not** modify the datasets, manifests, splits, notebooks, Git history, or checkpoints.

**Prerequisites:** Stage 1 original patch is already copied into `MyDrive/Document Unwarping/`; download `document_unwarping_stage1_hotfix.zip` from the conversation, then choose it in Cell 2's upload dialog.

## Cell 1: Mount Google Drive and locate the repository

In [2]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Document Unwarping")
assert ROOT.is_dir(), f"Repository not found: {ROOT}"
assert (ROOT / "requirements-stage1.txt").is_file(), "Original Stage 1 patch not found in project root"
assert (ROOT / "dataset/_metadata/uvdoc_final_manifest.csv").is_file(), "Dataset manifests not found"
os.environ["DOCUMENT_UNWARPING_ROOT"] = str(ROOT)
print("Repository:", ROOT)
print("Stage 1 code found:", (ROOT / "data/manifests.py").is_file())

Mounted at /content/drive
Repository: /content/drive/MyDrive/Document Unwarping
Stage 1 code found: True


## Cell 2: Upload and safely apply the hotfix (once)

Choose `document_unwarping_stage1_hotfix.zip`. If already applied, unchanged files are skipped. Before replacing any changed file, this cell backs up its previous version under `runs/stage1_hotfix_backups/` in Drive.

In [3]:
from google.colab import files
from datetime import datetime, timezone
from hashlib import sha256
from io import BytesIO
from zipfile import ZipFile
import shutil

HOTFIX = "document_unwarping_stage1_hotfix.zip"
allowed = {
    "data/dir300.py", "data/docunet.py", "data/images.py", "data/uvdoc.py",
    "documentation/stage1_environment_fix.md",
    "scripts/validate_datasets.py", "tests/test_optional_dependencies.py",
}

uploaded = files.upload()
assert HOTFIX in uploaded, f"Please upload the exact file {HOTFIX}"
backup_dir = ROOT / "runs" / "stage1_hotfix_backups" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
replaced, already_current, added = [], [], []
with ZipFile(BytesIO(uploaded[HOTFIX])) as archive:
    names = set(archive.namelist())
    assert names == allowed, f"Unexpected archive contents: {sorted(names ^ allowed)}"
    for name in sorted(allowed):
        target = ROOT / name
        new_bytes = archive.read(name)
        if target.is_file() and sha256(target.read_bytes()).digest() == sha256(new_bytes).digest():
            already_current.append(name)
            continue
        if target.is_file():
            backup = backup_dir / name
            backup.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, backup)
            replaced.append(name)
        else:
            added.append(name)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(new_bytes)

print("Updated existing:", replaced)
print("Added:", added)
print("Already up to date:", already_current)
print("Backups:", backup_dir if replaced else "Not needed")

Saving document_unwarping_stage1_hotfix.zip to document_unwarping_stage1_hotfix.zip
Updated existing: ['data/dir300.py', 'data/docunet.py', 'data/uvdoc.py', 'scripts/validate_datasets.py']
Added: ['data/images.py', 'documentation/stage1_environment_fix.md', 'tests/test_optional_dependencies.py']
Already up to date: []
Backups: /content/drive/MyDrive/Document Unwarping/runs/stage1_hotfix_backups/20261121T094909Z


## Cell 3: Install dependencies in Colab (not local Anaconda)

Colab uses its own Python environment. This does not repair or change your Windows `base` environment.

In [4]:
%pip install -q -r "/content/drive/MyDrive/Document Unwarping/requirements-stage1.txt"

## Cell 4: Verify compatible imports

In [5]:
import numpy, h5py, PIL, yaml
print("NumPy:", numpy.__version__)
print("h5py:", h5py.__version__)
print("Pillow:", PIL.__version__)
print("PyYAML:", yaml.__version__)

NumPy: 2.4.6
h5py: 3.14.0
Pillow: 11.3.0
PyYAML: 6.0.3


## Cell 5: Run Stage 1 tests

In [6]:
import subprocess, sys

test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    cwd=str(ROOT), text=True, capture_output=True
)
print(test_result.stdout)
if test_result.stderr:
    print("STDERR:\n", test_result.stderr)
print("Test exit code:", test_result.returncode)
assert test_result.returncode == 0, "Unit tests failed. Inspect the messages above."

.......................                                                  [100%]
23 passed in 10.60s

Test exit code: 0


## Cell 6: Validate all four manifests and the saved UVDoc split (metadata-only)

This check should work even if UVDoc Final is still only partially extracted.

In [7]:
from datetime import datetime, timezone

RUN_DIR = ROOT / "runs/data_validation" / ("stage1_colab_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
manifest_dir = RUN_DIR / "metadata_only"
result = subprocess.run(
    [sys.executable, "scripts/validate_datasets.py", "--root", str(ROOT), "--output", str(manifest_dir)],
    cwd=str(ROOT), text=True, capture_output=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)
print("Metadata exit code:", result.returncode)
print((manifest_dir / "report.md").read_text(encoding="utf-8") if (manifest_dir / "report.md").is_file() else "No report")
assert result.returncode == 0, "Metadata validation failed. Inspect report and errors."

Dataset validation report: /content/drive/MyDrive/Document Unwarping/runs/data_validation/stage1_colab_20261121T094937Z/metadata_only/report.md
uvdoc_final: 20000 rows, PASS, physical files not checked
uvdoc_benchmark: 50 rows, PASS, physical files not checked
docunet: 130 rows, PASS, physical files not checked
dir300: 300 rows, PASS, physical files not checked
saved UVDoc split: PASS

Metadata exit code: 0
# Stage 1 dataset validation

This report checks the existing EDA manifests without modifying source data.
No neural inference, benchmark scores, or geometric performance results are implied.

Project root: `/content/drive/MyDrive/Document Unwarping`

## Inventory

| Dataset | Records | Groups | Integrity | Files checked | Files unavailable |
|---|---:|---:|---|---:|---:|
| UVDoc Final | 20,000 | 1,008 | PASS | Not checked | Not checked |
| UVDoc Benchmark | 50 | 50 | PASS | Not checked | Not checked |
| DocUNet | 130 | 65 | PASS | Not checked | Not checked |
| DIR300 | 300 | 300 | 

## Cell 7: Check three real file paths and decode two real samples per dataset

Results may include `UNAVAILABLE` if a needed UVDoc Final file has not been staged. This is distinct from a corrupt manifest. For this script: 0 = success, 2 = some physical files unavailable, 3 = HDF5 environment error, 1 = structural or content validation failure.

In [8]:
smoke_dir = RUN_DIR / "physical_smoke"
smoke = subprocess.run(
    [sys.executable, "scripts/validate_datasets.py",
     "--root", str(ROOT), "--check-files", "--max-files-per-dataset", "3",
     "--smoke-read", "2", "--output", str(smoke_dir)],
    cwd=str(ROOT), text=True, capture_output=True
)
print(smoke.stdout)
if smoke.stderr:
    print("STDERR:\n", smoke.stderr)
print("Physical check exit code:", smoke.returncode)
print("Report:", smoke_dir / "report.md")
if (smoke_dir / "report.md").exists():
    print((smoke_dir / "report.md").read_text(encoding="utf-8"))
if (smoke_dir / "smoke.csv").exists():
    print("\nPer-sample decoding checks:")
    print((smoke_dir / "smoke.csv").read_text(encoding="utf-8")[:6000])
if smoke.returncode == 2:
    print("Some assets are UNAVAILABLE in this runtime, often because UVDoc Final is not fully staged.")
elif smoke.returncode == 3:
    print("ENVIRONMENT_ERROR: provide the smoke.csv and environment versions.")
elif smoke.returncode != 0:
    print("Validation failed: inspect issues.csv and smoke.csv.")

Dataset validation report: /content/drive/MyDrive/Document Unwarping/runs/data_validation/stage1_colab_20261121T094937Z/physical_smoke/report.md
uvdoc_final: 20000 rows, PASS, 8 unavailable physical files
uvdoc_benchmark: 50 rows, PASS, 0 unavailable physical files
docunet: 130 rows, PASS, 0 unavailable physical files
dir300: 300 rows, PASS, 0 unavailable physical files
saved UVDoc split: PASS
smoke read: {'PASS': 6, 'UNAVAILABLE': 2, 'FAIL': 0, 'ENVIRONMENT_ERROR': 0}

Physical check exit code: 2
Report: /content/drive/MyDrive/Document Unwarping/runs/data_validation/stage1_colab_20261121T094937Z/physical_smoke/report.md
# Stage 1 dataset validation

This report checks the existing EDA manifests without modifying source data.
No neural inference, benchmark scores, or geometric performance results are implied.

Project root: `/content/drive/MyDrive/Document Unwarping`

## Inventory

| Dataset | Records | Groups | Integrity | Files checked | Files unavailable |
|---|---:|---:|---|---:|--

## Cell 8. Stage only the missing UVDoc Final *files*

In [9]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import shutil
import sys

ROOT = Path("/content/drive/MyDrive/Document Unwarping")
ARCHIVE = ROOT / "dataset/UVDoc/_archives/UVDoc_final.zip"
STAGING = Path("/content/uvdoc_runtime")

sys.path.insert(0, str(ROOT))

from data.manifests import load_records
from data.paths import PathResolver

assert ARCHIVE.is_file(), f"Archive not found: {ARCHIVE}"

resolver = PathResolver(ROOT)
records = load_records("uvdoc_final", ROOT)[:3]

required = set()

for record in records:
    paths = [record.input_path, *record.annotation_paths.values()]
    for path in paths:
        if resolver.find(path) is None:
            required.add(path)

print("Missing UVDoc Final files:", len(required))

with ZipFile(ARCHIVE) as archive:
    for logical_path in sorted(required):
        assert logical_path.startswith("dataset/UVDoc/UVDoc_final/")

        member = logical_path.removeprefix("dataset/UVDoc/")
        parts = PurePosixPath(member).parts

        assert parts[0] == "UVDoc_final"
        assert ".." not in parts
        assert "\\" not in member

        info = archive.getinfo(member)
        assert not info.is_dir()

        destination = STAGING.joinpath(*parts)
        destination.parent.mkdir(parents=True, exist_ok=True)

        temporary = destination.with_suffix(destination.suffix + ".partial")

        try:
            with archive.open(info) as source, temporary.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024 * 1024)

            assert temporary.stat().st_size == info.file_size
            temporary.replace(destination)
        finally:
            if temporary.exists():
                temporary.unlink()

        print("Staged:", member)

resolver = PathResolver(ROOT)

remaining = [
    path for path in required
    if resolver.find(path) is None
]

assert not remaining, f"Files still unavailable: {remaining}"

print("\nUVDoc Final representative files are now available.")

Missing UVDoc Final files: 8
Staged: UVDoc_final/grid2d/00_00000_1_0_0.mat
Staged: UVDoc_final/grid2d/00_00000_1_0_1.mat
Staged: UVDoc_final/grid2d/00_00000_1_1_0.mat
Staged: UVDoc_final/img/00000.png
Staged: UVDoc_final/img/00001.png
Staged: UVDoc_final/img/00002.png
Staged: UVDoc_final/metadata_sample/00000.json
Staged: UVDoc_final/metadata_sample/00001.json

UVDoc Final representative files are now available.


## Cell 9. Repeat the physical validation

In [10]:
import subprocess
import sys
from datetime import datetime, timezone

output = (
    ROOT / "runs/data_validation"
    / ("stage1_final_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
)

result = subprocess.run(
    [
        sys.executable,
        "scripts/validate_datasets.py",
        "--root", str(ROOT),
        "--check-files",
        "--max-files-per-dataset", "3",
        "--smoke-read", "2",
        "--output", str(output),
    ],
    cwd=str(ROOT),
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)
print("\n" + (output / "report.md").read_text(encoding="utf-8"))

assert result.returncode == 0, (
    "Physical validation has remaining issues. "
    "Inspect the generated report."
)

Dataset validation report: /content/drive/MyDrive/Document Unwarping/runs/data_validation/stage1_final_20261121T095429Z/report.md
uvdoc_final: 20000 rows, PASS, 0 unavailable physical files
uvdoc_benchmark: 50 rows, PASS, 0 unavailable physical files
docunet: 130 rows, PASS, 0 unavailable physical files
dir300: 300 rows, PASS, 0 unavailable physical files
saved UVDoc split: PASS
smoke read: {'PASS': 8, 'UNAVAILABLE': 0, 'FAIL': 0, 'ENVIRONMENT_ERROR': 0}

Exit code: 0

# Stage 1 dataset validation

This report checks the existing EDA manifests without modifying source data.
No neural inference, benchmark scores, or geometric performance results are implied.

Project root: `/content/drive/MyDrive/Document Unwarping`

## Inventory

| Dataset | Records | Groups | Integrity | Files checked | Files unavailable |
|---|---:|---:|---|---:|---:|
| UVDoc Final | 20,000 | 1,008 | PASS | 12 | 0 |
| UVDoc Benchmark | 50 | 50 | PASS | 12 | 0 |
| DocUNet | 130 | 65 | PASS | 6 | 0 |
| DIR300 | 300 | 3

## What to share

Send the Cell 5 test output, the Cell 7 console output, and, ideally, `runs/data_validation/<run_id>/physical_smoke/report.md` plus `smoke.csv`. Do not upload any raw dataset archives or model weights.

**Do not run local Conda/Powershell repair commands for this Colab workflow.**